In [4]:
import os
import sqlite3
import pandas as pd

# 1. Relative file paths
raw_data_path = '../data/raw/E_Commerce_Dataset.csv'
db_path = '../data/processed/ecommerce_churn.db'

# Ensure the target directory exists
os.makedirs('../data/processed', exist_ok=True)

# 2. Load the dataset with Pandas
print("Loading E-Commerce dataset...")
df = pd.read_csv(raw_data_path)

# Print dataset dimensions
print(f"Dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

# 3. Clean column names for SQL standard (lowercase and underscores)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# 4. Connect to SQLite database and save data
conn = sqlite3.connect(db_path)

# Save DataFrame as 'customers' SQL table
df.to_sql('customers', conn, if_exists='replace', index=False)

print(f"\nSQLite database created successfully at: {db_path}")
print("Table 'customers' created successfully.")

# 5. Test initial SQL aggregation
query = """
SELECT 
    CASE 
        WHEN days_since_last_purchase > 30 THEN 'Inactive / At Risk'
        ELSE 'Active'
    END AS customer_status,
    COUNT(*) AS total_customers,
    ROUND(AVG(total_spend), 2) AS avg_total_spend,
    ROUND(AVG(items_purchased), 2) AS avg_items_purchased,
    ROUND(AVG(days_since_last_purchase), 1) AS avg_days_since_purchase
FROM customers
GROUP BY customer_status;
"""

result = pd.read_sql_query(query, conn)
print("\n--- Initial Customer Status Overview ---")
print(result)

Loading E-Commerce dataset...
Dataset dimensions: 350 rows, 11 columns

SQLite database created successfully at: ../data/processed/ecommerce_churn.db
Table 'customers' created successfully.

--- Initial Customer Status Overview ---
      customer_status  total_customers  avg_total_spend  avg_items_purchased  \
0              Active              226           969.30                13.65   
1  Inactive / At Risk              124           619.52                10.69   

   avg_days_since_purchase  
0                     17.9  
1                     42.3  


In [5]:
query_membership = """
SELECT 
    membership_type,
    COUNT(customer_id) AS total_customers,
    ROUND(AVG(total_spend), 2) AS avg_spend,
    ROUND(AVG(items_purchased), 1) AS avg_items,
    ROUND(AVG(days_since_last_purchase), 1) AS avg_recency_days,
    ROUND(SUM(CASE WHEN days_since_last_purchase > 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS risk_rate_pct
FROM customers
GROUP BY membership_type
ORDER BY avg_spend DESC;
"""

df_membership = pd.read_sql_query(query_membership, conn)
print("--- Analysis by Membership Type ---")
print(df_membership)

--- Analysis by Membership Type ---
  membership_type  total_customers  avg_spend  avg_items  avg_recency_days  \
0            Gold              117    1311.14       17.6              17.9   
1          Silver              117     748.43       11.7              30.3   
2          Bronze              116     473.39        8.5              31.6   

   risk_rate_pct  
0           5.13  
1          49.57  
2          51.72  


In [6]:
query_rfm = """
WITH rfm_scores AS (
    SELECT 
        customer_id,
        membership_type,
        total_spend,
        items_purchased,
        days_since_last_purchase,
        NTILE(4) OVER (ORDER BY days_since_last_purchase DESC) AS r_score,
        NTILE(4) OVER (ORDER BY items_purchased ASC) AS f_score,
        NTILE(4) OVER (ORDER BY total_spend ASC) AS m_score
    FROM customers
)
SELECT 
    CASE 
        WHEN (r_score + f_score + m_score) >= 10 THEN 'Champions / Loyal'
        WHEN (r_score + f_score + m_score) BETWEEN 7 AND 9 THEN 'Potential Loyalists'
        WHEN (r_score + f_score + m_score) BETWEEN 5 AND 6 THEN 'At Risk / Need Attention'
        ELSE 'Lost / Hibernating'
    END AS customer_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_spend), 2) AS avg_spend,
    ROUND(AVG(days_since_last_purchase), 1) AS avg_recency
FROM rfm_scores
GROUP BY customer_segment
ORDER BY avg_spend DESC;
"""

df_rfm = pd.read_sql_query(query_rfm, conn)
print("--- RFM Customer Segmentation Summary ---")
print(df_rfm)

--- RFM Customer Segmentation Summary ---
           customer_segment  customer_count  avg_spend  avg_recency
0         Champions / Loyal             112    1243.31         14.4
1       Potential Loyalists              64     963.71         21.6
2  At Risk / Need Attention             115     586.59         37.7
3        Lost / Hibernating              59     466.07         33.4


In [ ]:
# Query to enrich original dataset with RFM segments
export_query = """
WITH rfm_scores AS (
    SELECT 
        customer_id,
        gender,
        age,
        city,
        membership_type,
        total_spend,
        items_purchased,
        average_rating,
        discount_applied,
        days_since_last_purchase,
        satisfaction_level,
        NTILE(4) OVER (ORDER BY days_since_last_purchase DESC) AS r_score,
        NTILE(4) OVER (ORDER BY items_purchased ASC) AS f_score,
        NTILE(4) OVER (ORDER BY total_spend ASC) AS m_score
    FROM customers
)
SELECT 
    customer_id,
    gender,
    age,
    city,
    membership_type,
    total_spend,
    items_purchased,
    average_rating,
    discount_applied,
    days_since_last_purchase,
    satisfaction_level,
    CASE 
        WHEN (r_score + f_score + m_score) >= 10 THEN 'Champions / Loyal'
        WHEN (r_score + f_score + m_score) BETWEEN 7 AND 9 THEN 'Potential Loyalists'
        WHEN (r_score + f_score + m_score) BETWEEN 5 AND 6 THEN 'At Risk / Need Attention'
        ELSE 'Lost / Hibernating'
    END AS customer_segment
FROM rfm_scores;
"""

# Load enriched query into DataFrame
df_export = pd.read_sql_query(export_query, conn)

# Export to CSV
output_csv_path = '../data/processed/ecommerce_rfm_segments.csv'
df_export.to_csv(output_csv_path, index=False)

# Close database connection
conn.close()

print(f"File successfully exported to: {output_csv_path}")
print(f"Total exported records: {len(df_export)}")